## Data Loading and Packages

In [ ]:
### Load libraries
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score

DATA_PATH = Path("..", "..", "data", "MLB_2021-2025.csv")

PITCHER_COL = "pitcher"
PITCH_TYPE_COL = "pitch_type"
SEASON_COL = "game_year"

# Same modeling-scope exclusion as Stuff+ (pitching_plus/scripts/stuff.py) -- pitchouts,
# unknowns, eephus, forkball, knuckleball, slow curve, screwball, generic "fastball"
JUNK_PITCH_TYPES = {"FA", "EP", "FO", "KN", "CS", "SC", "PO", "UN"}

## Load data
cols_used = [
    "player_name",
    "pitcher",
    "pitch_type",
    "balls",
    "strikes",
    "outs_when_up",
    "on_1b",
    "on_2b",
    "on_3b",
    "stand",
    "p_throws",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "delta_pitcher_run_exp",
]

load_cols = cols_used + [SEASON_COL]

# usecols avoids reading all ~119 raw Statcast columns before subsetting.
# NOTE: unlike stuff.ipynb, we do NOT dropna() at load time here -- on_1b/
# on_2b/on_3b are legitimately NaN when a base is empty, not missing data.
data = pd.read_csv(
    DATA_PATH,
    usecols=load_cols
)

data.head()

## Scope & Filtering

In [9]:
# ============================================================
# SCOPE
# ============================================================
# Drop pitch types out of scope -- same modeling-scope decision as Stuff+.

df = data.copy()
df = df[~df[PITCH_TYPE_COL].isin(JUNK_PITCH_TYPES)].copy()

# Drop rows without a usable location or run-value target: automatic
# ball/strike calls (pitch-clock violations -- no pitch location at all),
# pitchouts, and the small remainder missing plate_x/plate_z or
# delta_pitcher_run_exp for other reasons (~0.3% of pitches).
# on_1b/on_2b/on_3b are NOT included here -- NaN there means "base empty",
# a real game state, not a missing-data problem.
required_cols = [
    PITCHER_COL, PITCH_TYPE_COL, SEASON_COL, "player_name",
    "balls", "strikes", "outs_when_up", "stand", "p_throws",
    "plate_x", "plate_z", "sz_top", "sz_bot",
    "delta_pitcher_run_exp",
]
df = df.dropna(subset=required_cols).copy()

print(f"{len(df):,} / {len(data):,} pitches in scope")

3,540,371 / 3,565,743 pitches in scope


## Feature Engineering

#### RE288 Situation State

`delta_pitcher_run_exp` (Statcast's per-pitch run value, sign-flipped so positive
favors the pitcher; verified `delta_pitcher_run_exp == -delta_run_exp` exactly)
is itself computed from a run-expectancy table conditioned on both the 24
base-out states *and* the ball-strike count: RE288 (24 x 12 = 288 states),
not just RE24. Verified directly on this data: holding base-out state fixed at
empty bases / 0 outs, a called strike is worth -0.040 runs at a 0-0 count but
-0.344 runs at 3-2, and a ball is worth +0.040 runs at 0-0 but +0.361 runs at
3-2. So rather than rebuild a run-expectancy matrix from half-inning outcomes,
Location+ uses `delta_pitcher_run_exp` directly as its RE288-based per-pitch
target, and only needs to model *where a pitch was worth doing that*.

In [10]:
# ============================================================
# BASE-OUT-COUNT STATE (RE288), AS AN EXPLICIT FEATURE
# ============================================================
# Included as its own categorical id (in addition to the raw components
# below) so the model can route on "the situation" as a single unit where
# that helps, on top of whatever it learns from the individual components.

df["on_1b_occupied"] = df["on_1b"].notna().astype(int)
df["on_2b_occupied"] = df["on_2b"].notna().astype(int)
df["on_3b_occupied"] = df["on_3b"].notna().astype(int)

# 8 base combos x 3 out counts = 24 base-out states;
# 4 ball counts x 3 strike counts = 12 count states;
# 24 x 12 = 288 combined situation states.
base_state = df["on_1b_occupied"] * 4 + df["on_2b_occupied"] * 2 + df["on_3b_occupied"]
count_state = df["balls"] * 3 + df["strikes"]
df["re288_state"] = (df["outs_when_up"] * 8 + base_state) * 12 + count_state


# ============================================================
# LOCATION, NORMALIZED FOR BATTER STRIKE ZONE + HANDEDNESS
# ============================================================

# 0 = bottom of THIS batter's strike zone, 1 = top -- makes vertical location
# comparable across batters of different heights/stances.
df["plate_z_rel"] = (df["plate_z"] - df["sz_bot"]) / (df["sz_top"] - df["sz_bot"])

# Statcast's plate_x is from the catcher's view, so the same value means
# "inside" for one batter side and "outside" for the other. Flip so positive
# is arm-side / negative is glove-side regardless of batter stand. Sign
# convention is arbitrary but consistent; raw plate_x and stand are kept too
# so the model isn't dependent on this specific transform being the useful one.
df["plate_x_armside"] = np.where(df["stand"] == "R", df["plate_x"], -df["plate_x"])


# ============================================================
# HANDEDNESS
# ============================================================

df["stand_R"] = (df["stand"] == "R").astype(int)
df["p_throws_R"] = (df["p_throws"] == "R").astype(int)
df["platoon_matchup"] = (df["stand"] != df["p_throws"]).astype(int)  # 1 = batter has the platoon advantage

df[["re288_state", "plate_z_rel", "plate_x_armside", "stand_R", "p_throws_R", "platoon_matchup"]].describe()

,re288_state,plate_z_rel,plate_x_armside,stand_R,p_throws_R,platoon_matchup
count,3.540371e+06,3.540371e+06,3.540371e+06,3.540371e+06,3.540371e+06,3.540371e+06
mean,1.180009e+02,3.904524e-01,1.700604e-01,5.798788e-01,7.245775e-01,5.416424e-01
std,8.761353e+01,5.454289e-01,8.195731e-01,4.935782e-01,4.467270e-01,4.982630e-01
min,0.000000e+00,-4.106408e+00,-8.825433e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.400000e+01,3.592648e-02,-3.782858e-01,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.040000e+02,3.934036e-01,1.734061e-01,1.000000e+00,1.000000e+00,1.000000e+00
75%,1.950000e+02,7.487064e-01,7.172821e-01,1.000000e+00,1.000000e+00,1.000000e+00
max,2.870000e+02,6.280768e+00,1.136539e+01,1.000000e+00,1.000000e+00,1.000000e+00


## Model: Expected Run Value by Location

Unlike Stuff+ (a standardized index over physics, no outcome data), Location+ is
an actual trained model: predict `delta_pitcher_run_exp` from where/when the
pitch was thrown, per pitch type (gradient-boosted trees, since the
location-value surface is highly non-linear: e.g. "just off the plate away" is
good, "middle-away" is bad, "further off the plate" is neutral-again once it's
an obvious ball).

**Out-of-fold prediction is the point, not an accuracy nicety here.** A single
pitch's realized `delta_pitcher_run_exp` is mostly noise (a well-located pitch
can still get bloop-hit; a mistake can be missed). Location+ wants the model's
*expectation* for that (pitch_type, location, situation, handedness) combo, not
the realized outcome, so every pitch's `location_run_value` must come from a
fold that never trained on that pitch (5-fold CV). The model refit on all
in-scope data afterward is what a future `location.py` would use to score new
pitches going forward.

In [11]:
# ============================================================
# CONFIGURATION
# ============================================================

LOCATION_FEATURES = [
    "plate_x", "plate_z_rel", "plate_x_armside",
    "balls", "strikes",
    "on_1b_occupied", "on_2b_occupied", "on_3b_occupied", "outs_when_up",
    "re288_state",
    "stand_R", "p_throws_R", "platoon_matchup",
]

TARGET_COL = "delta_pitcher_run_exp"

MIN_GROUP_SIZE_FOR_MODEL = 5000  # same scope gate as Stuff+
N_FOLDS = 5


# ============================================================
# PER-PITCH-TYPE MODEL, OUT-OF-FOLD SCORING
# ============================================================

df["location_run_value"] = np.nan
location_models = {}

for ptype, type_group in df.groupby(PITCH_TYPE_COL, observed=True):

    if len(type_group) < MIN_GROUP_SIZE_FOR_MODEL:
        continue

    X = type_group[LOCATION_FEATURES].to_numpy()
    y = type_group[TARGET_COL].to_numpy()

    model = HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=300,
        l2_regularization=1.0,
        random_state=42,
    )

    kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    oof_pred = cross_val_predict(model, X, y, cv=kfold, n_jobs=-1)
    df.loc[type_group.index, "location_run_value"] = oof_pred

    # Refit on all in-scope data for this pitch type for future scoring.
    model.fit(X, y)
    location_models[ptype] = model

print(f"scored {df['location_run_value'].notna().sum():,} / {len(df):,} pitches "
      f"across {len(location_models)} pitch types")

scored 3,540,371 / 3,540,371 pitches across 10 pitch types


## Model Quality Check

Expect low R^2 here. Most single-pitch run-value variance is swing decision
and contact quality, which this model has no access to (it only sees where/when
the pitch was thrown, not the swing). A small but real R^2, consistent in sign
and magnitude across pitch types, is the right signature of "there is usable
location signal, correctly not overfit to single-pitch noise."

In [12]:
print(f"{'pitch_type':<12}{'n':>10}{'r2':>10}{'corr':>10}")
for ptype, type_group in df.dropna(subset=["location_run_value"]).groupby(PITCH_TYPE_COL, observed=True):
    r2 = r2_score(type_group[TARGET_COL], type_group["location_run_value"])
    corr = type_group[TARGET_COL].corr(type_group["location_run_value"])
    print(f"{ptype:<12}{len(type_group):>10,}{r2:>10.4f}{corr:>10.4f}")

pitch_type           n        r2      corr
CH             382,610    0.0335    0.1834
CU             241,554    0.0317    0.1784
FC             273,054    0.0313    0.1769
FF           1,168,894    0.0429    0.2073
FS              83,821    0.0329    0.1818
KC              71,720    0.0246    0.1572
SI             550,489    0.0449    0.2120
SL             550,770    0.0316    0.1782
ST             201,003    0.0318    0.1787
SV              16,456    0.0197    0.1403


## Location+ Scaling (100+ Scale)

Same calibration approach as Stuff+, for the same reasons (genuine ratio scale,
"100 = league average for that pitch type/season," dual-level so pitch-level
scores stay on the same scale as the pitcher-season headline figure; see
stuff.ipynb): z-score `location_run_value` against the *reliable* (>=20 pitch)
pitcher-season population per (pitch_type, season), then
`100 * exp(k * z)`, renormalized so that population's mean is exactly 100.

Two outputs, both signed so positive favors the pitcher:
1. **`location_run_value`**: the actual expected change in run value (runs), pitch-level.
2. **`location_plus` / `pitch_location_plus`**: the 100+ scaled score, at the
   pitcher-season and pitch level respectively.

In [13]:
# ============================================================
# LOCATION+ SCORING: CONFIGURATION
# ============================================================

MIN_PITCHES_FOR_SCORE = 20  # same reliability gate as Stuff+
LOCATION_SCALE_K = 0.10


# ============================================================
# AGGREGATE TO (pitcher, pitch_type, season)
# ============================================================

scored = df.dropna(subset=["location_run_value"]).copy()

pitcher_agg = (
    scored
    .groupby([PITCHER_COL, PITCH_TYPE_COL, SEASON_COL, "player_name"], observed=True)["location_run_value"]
    .agg(mean_location_value="mean", n_pitches="count")
    .reset_index()
)


# ============================================================
# CALIBRATION: league reference distribution, per (pitch_type, season)
# ============================================================

reliable = pitcher_agg[pitcher_agg["n_pitches"] >= MIN_PITCHES_FOR_SCORE].copy()

calibration = (
    reliable
    .groupby([PITCH_TYPE_COL, SEASON_COL])["mean_location_value"]
    .agg(agg_mu="mean", agg_sigma="std")
    .reset_index()
)
reliable = reliable.merge(calibration, on=[PITCH_TYPE_COL, SEASON_COL], how="left")
reliable["agg_z"] = (reliable["mean_location_value"] - reliable["agg_mu"]) / reliable["agg_sigma"]
reliable["raw_ratio"] = np.exp(LOCATION_SCALE_K * reliable["agg_z"])

raw_ratio_mean = (
    reliable
    .groupby([PITCH_TYPE_COL, SEASON_COL])["raw_ratio"]
    .mean()
    .rename("raw_ratio_mean")
    .reset_index()
)
calibration = calibration.merge(raw_ratio_mean, on=[PITCH_TYPE_COL, SEASON_COL], how="left")

del reliable


# ============================================================
# APPLY THE SAME CALIBRATION TO BOTH LEVELS -- SHARED SCALE, SHARED "100"
# ============================================================

pitcher_agg = pitcher_agg.merge(calibration, on=[PITCH_TYPE_COL, SEASON_COL], how="left")
pitcher_agg["location_plus"] = 100 * np.exp(
    LOCATION_SCALE_K * (pitcher_agg["mean_location_value"] - pitcher_agg["agg_mu"]) / pitcher_agg["agg_sigma"]
) / pitcher_agg["raw_ratio_mean"]
pitcher_agg["reliable"] = pitcher_agg["n_pitches"] >= MIN_PITCHES_FOR_SCORE

scored = scored.merge(calibration, on=[PITCH_TYPE_COL, SEASON_COL], how="left")
scored["pitch_location_plus"] = 100 * np.exp(
    LOCATION_SCALE_K * (scored["location_run_value"] - scored["agg_mu"]) / scored["agg_sigma"]
) / scored["raw_ratio_mean"]


# ============================================================
# OUTPUTS
# ============================================================
#
# Pitch-level Location+ (every individual pitch -- feed this into Pitching+,
# which combines Stuff+ and Location+ per pitch):
#
#     scored["location_run_value"]   -- raw expected run value (runs)
#     scored["pitch_location_plus"]  -- 100+ scaled
#
# Pitcher x pitch_type x season Location+ (headline reporting figure,
# filtered to reliable pitcher-seasons):
#
#     pitcher_location_plus

pitcher_location_plus = (
    pitcher_agg[pitcher_agg["reliable"]]
    .sort_values("location_plus", ascending=False)
    .reset_index(drop=True)
)

pitcher_location_plus.head(15)

,pitcher,pitch_type,game_year,player_name,mean_location_value,n_pitches,agg_mu,agg_sigma,raw_ratio_mean,location_plus,reliable
0,543901,CH,2023,"Weber, Ryan",0.022045,26,-0.004234,0.007308,1.004885,142.578260,True
1,594577,FC,2023,"Mayers, Mike",0.018277,42,-0.000535,0.005434,1.004989,140.662757,True
2,595014,SI,2022,"Treinen, Blake",0.021886,22,-0.000491,0.006835,1.004823,138.067521,True
3,641726,FC,2024,"Jefferies, Daulton",0.017073,41,-0.000660,0.005542,1.004914,137.032627,True
4,571539,FF,2024,"Carasiti, Matt",0.017199,30,-0.001837,0.005990,1.004781,136.759810,True
5,687396,FF,2024,"Headrick, Brent",0.016990,23,-0.001837,0.005990,1.004781,136.283620,True
6,608650,SI,2025,"Enns, Dietrich",0.021630,23,0.001827,0.006299,1.004891,136.274055,True
7,674003,SL,2024,"Bradford, Cody",0.018296,84,0.001491,0.005356,1.004855,136.190399,True
8,681151,SI,2025,"Murray, Jayden",0.021266,39,0.001827,0.006299,1.004891,135.488092,True
9,606965,SL,2022,"Devenski, Chris",0.018414,29,0.000642,0.005806,1.004902,135.145864,True


In [14]:
# ============================================================
# TOP 10 PER PITCH TYPE (ONE SEASON PER PITCHER -- THEIR BEST)
# ============================================================

best_season_idx = (
    pitcher_location_plus
    .groupby([PITCHER_COL, PITCH_TYPE_COL])["location_plus"]
    .idxmax()
)

best_season = pitcher_location_plus.loc[best_season_idx]

top10_by_type = (
    best_season
    .sort_values([PITCH_TYPE_COL, "location_plus"], ascending=[True, False])
    .groupby(PITCH_TYPE_COL)
    .head(10)
    [[PITCH_TYPE_COL, "player_name", SEASON_COL, "n_pitches", "location_plus"]]
    .reset_index(drop=True)
)

top10_by_type["rank"] = top10_by_type.groupby(PITCH_TYPE_COL).cumcount() + 1
top10_by_type = top10_by_type[[PITCH_TYPE_COL, "rank", "player_name", SEASON_COL, "n_pitches", "location_plus"]]

display(top10_by_type[top10_by_type["pitch_type"] == "FF"])

,pitch_type,rank,player_name,game_year,n_pitches,location_plus
30,FF,1,"Carasiti, Matt",2024,30,136.759810
31,FF,2,"Headrick, Brent",2024,23,136.283620
32,FF,3,"Miller, Bobby",2025,31,132.802442
33,FF,4,"Speier, Gabe",2021,26,129.332388
34,FF,5,"Feyereisen, J.P.",2025,28,128.371681
35,FF,6,"Wells, Alex",2022,28,126.831667
36,FF,7,"Ivey, Tyler",2021,40,126.531284
37,FF,8,"Avila, Pedro",2022,33,126.333390
38,FF,9,"Sulser, Cole",2025,153,125.971697
39,FF,10,"Crawford, Kutter",2021,28,125.647108


## Overall Leaderboard

A pitcher's per-pitch-type Location+ above can be a small, noisy sample (22
sinkers isn't a meaningful read of a pitcher's command). This "final" score
instead averages the raw `location_run_value` (already directly comparable
in run units across pitch types, no per-type calibration needed) across
*every* in-scope pitch a pitcher threw that season, regardless of type. It's
naturally usage-weighted (a pitch type thrown more often contributes more rows
to the average), and reliability is judged on the season's total pitch count,
calibrated against all reliable pitcher-seasons that year (100 = league
average across all pitch types, same season). This eventually runs pitch-by-
pitch too (feeding Pitching+), but the season aggregate is the "how good is
this pitcher's command, full stop" read. Top 10 pitcher-seasons, one row per
pitcher (their best season).

In [16]:
# ============================================================
# OVERALL LOCATION+ (per pitcher-season, across the whole arsenal)
# ============================================================

# A full season's worth of pitches is the bar for "reliable" here -- much
# higher than the per-pitch-type gate (20), since a few dozen pitches total
# in a season is too thin to call a real read on a pitcher's command.
MIN_PITCHES_FOR_SEASON_SCORE = 100

pitcher_season_agg = (
    scored
    .groupby([PITCHER_COL, SEASON_COL, "player_name"], observed=True)["location_run_value"]
    .agg(mean_location_value="mean", n_pitches="count")
    .reset_index()
)

reliable_season = pitcher_season_agg[pitcher_season_agg["n_pitches"] >= MIN_PITCHES_FOR_SEASON_SCORE].copy()

season_calibration = (
    reliable_season
    .groupby(SEASON_COL)["mean_location_value"]
    .agg(agg_mu="mean", agg_sigma="std")
    .reset_index()
)
reliable_season = reliable_season.merge(season_calibration, on=SEASON_COL, how="left")
reliable_season["agg_z"] = (reliable_season["mean_location_value"] - reliable_season["agg_mu"]) / reliable_season["agg_sigma"]
reliable_season["raw_ratio"] = np.exp(LOCATION_SCALE_K * reliable_season["agg_z"])

season_raw_ratio_mean = (
    reliable_season
    .groupby(SEASON_COL)["raw_ratio"]
    .mean()
    .rename("raw_ratio_mean")
    .reset_index()
)
season_calibration = season_calibration.merge(season_raw_ratio_mean, on=SEASON_COL, how="left")

del reliable_season

pitcher_season_agg = pitcher_season_agg.merge(season_calibration, on=SEASON_COL, how="left")
pitcher_season_agg["location_plus"] = 100 * np.exp(
    LOCATION_SCALE_K * (pitcher_season_agg["mean_location_value"] - pitcher_season_agg["agg_mu"]) / pitcher_season_agg["agg_sigma"]
) / pitcher_season_agg["raw_ratio_mean"]
pitcher_season_agg["reliable"] = pitcher_season_agg["n_pitches"] >= MIN_PITCHES_FOR_SEASON_SCORE

pitcher_season_location_plus = (
    pitcher_season_agg[pitcher_season_agg["reliable"]]
    .sort_values("location_plus", ascending=False)
    .reset_index(drop=True)
)


# ============================================================
# TOP 10 OVERALL (ONE ROW PER PITCHER -- THEIR BEST SEASON)
# ============================================================

best_season_idx = pitcher_season_location_plus.groupby(PITCHER_COL)["location_plus"].idxmax()

top10_overall = (
    pitcher_season_location_plus
    .loc[best_season_idx]
    .sort_values("location_plus", ascending=False)
    .head(10)
    [["player_name", SEASON_COL, "n_pitches", "location_plus"]]
    .reset_index(drop=True)
)
top10_overall.insert(0, "rank", top10_overall.index + 1)

display(top10_overall)

,rank,player_name,game_year,n_pitches,location_plus
0,1,"Springs, Jeffrey",2023,211,130.481600
1,2,"Cobb, Alex",2024,242,128.647530
2,3,"Speier, Gabe",2021,105,128.496121
3,4,"Busenitz, Alan",2023,107,128.364004
4,5,"Milone, Tommy",2023,163,127.632090
5,6,"Milner, Hoby",2023,1027,127.465105
6,7,"Ryu, Hyun Jin",2022,395,127.225820
7,8,"Campbell, Isaiah",2025,109,126.986820
8,9,"Martin, Chris",2023,722,126.458223
9,10,"Newsome, Ljay",2021,276,126.045068
